In [ ]:
!git clone https://github.com/OpenMOSS/MOSS-TTS.git
%cd MOSS-TTS
!pip install datasets soundfile -q wandb
!apt-get install -y ffmpeg -q

In [ ]:
# MODEL_NAME = "OpenMOSS-Team/MOSS-TTS"                 # Delay 8B
MODEL_NAME = "OpenMOSS-Team/MOSS-TTS-Local-Transformer" # Local 1.7B

N_SAMPLES = 20
MIN_WORDS = 10
MAX_WORDS = 50
MAX_NEW_TOKENS = 200
OUTPUT_DIR = "/content/profiling_results_hf"

In [ ]:
import json, random, time
from pathlib import Path
import numpy as np
import soundfile as sf
import torch
from datasets import load_dataset
from transformers import AutoModel, AutoProcessor

import wandb

# required SDPA backend flags from official MOSS-TTS docs
torch.backends.cuda.enable_cudnn_sdp(False)
torch.backends.cuda.enable_flash_sdp(True)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'{device}')

In [ ]:
#sample runner: run through samples and collect correct metrics
def run_one_sample(model, processor, text, output_dir, idx):
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    batch = processor([[processor.build_user_message(text=text)]], mode='generation')
    inputs = {k: v.to(device) for k, v in batch.items()}

    #synchronize and reset GPU mem stats before generation
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    t0 = time.perf_counter()
    
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS)
    

    torch.cuda.synchronize()

    # get memory and timing stats
    totalS = time.perf_counter() - t0
    peakMB = torch.cuda.max_memory_allocated() / 1e6

    # decode audio and save
    messages  = processor.decode(out)
    audio = messages[0].audio_codes_list[0].cpu().float()
    sr = processor.model_config.sampling_rate
    audioDur = audio.shape[-1] / sr

    sf.write(str(Path(output_dir) / f'sample_{idx:03d}.wav'), audio.squeeze().numpy(), sr)

    return {'text': text, 'word_count': len(text.split()),
            'total_time_s': round(totalS, 3),
            'audio_dur_s': round(audioDur, 3),
            'rtf': round(totalS / audioDur if audioDur > 0 else 0, 4),
            'peak_gpu_mb': round(peakMB, 1)}

In [ ]:
# load dataset
ds = load_dataset('wikitext', 'wikitext-103-raw-v1', split='test')

texts = [row['text'].strip() for row in ds if MIN_WORDS < len(row['text'].split()) <= MAX_WORDS]
random.seed(42)

texts = random.sample(texts, min(N_SAMPLES, len(texts)))

In [ ]:
# model and processor loading
processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)

if hasattr(processor, 'audio_tokenizer'):
    processor.audio_tokenizer = processor.audio_tokenizer.to(device).eval()

model = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True, torch_dtype=torch.bfloat16).to(device).eval()

In [ ]:
# warmup 
warmup = processor([[processor.build_user_message(text='Hello warmup.')]], mode='generation')

with torch.no_grad():
    _ = model.generate(**{k: v.to(device) for k, v in warmup.items()}, max_new_tokens=200)

In [ ]:
out = Path(OUTPUT_DIR)
out.mkdir(parents=True, exist_ok=True)


wandbRun = wandb.init(project="hpml-final-project", name= f"hf-{MODEL_NAME}-inference-profile")

# for texts run sample and collect res
for i, text in enumerate(texts, 1):

    run = run_one_sample(model, processor, text, str(out/'wav'), i)

    print(f'[{i:2d}/{len(texts)}] {run["word_count"]:3d}w  'f'RTF={run["rtf"]:.3f}  dur={run["audio_dur_s"]:.1f}s  peak={run["peak_gpu_mb"]:.0f}MB')
    wandbRun.log({"text": text, "word_count": run["word_count"], "rtf": run["rtf"], "audio_dur_s": run["audio_dur_s"], "peak_gpu_mb": run["peak_gpu_mb"], "total_time_s": run["total_time_s"]})

wandbRun.finish()